# Prompt evolution

Runs one EA run from Python on HumanEvalFix, the same thing one run of the
study does (`python -m promptevo.study`). Needs the vLLM solver server from
`src/promptevo/README.md`, Ollama with the mutator model, and Docker for the
sandbox.

In [ ]:
import sys
sys.path.append("../src")

from promptevo.datasets import HumanEval
from promptevo.llm.models import mutator, solver
from promptevo.scoring.evaluation import Evaluator, Signal
from promptevo.scoring.experiment import (
    pick_by_validation,
    score_test_set,
    test_prompts,
)
from promptevo.search.evolution import Evolution, EvolutionConfig
from promptevo.search.mutations import MUTATIONS, MixedOperators
from promptevo.search.runlog import RunLog
from promptevo.search.selection import UniformSelector
from promptevo.search.templates import SEED_PROMPTS
from promptevo.study.design import split

## Settings

A unit is the study's own split of the 164 problems into 40 train, 30
validation and 94 test problems. The caches keep solver replies and sandbox
results, so running a cell again is fast. Point them at
`../results/study/` to reuse the study's.

In [ ]:
unit = 40
sets = split(unit)

cache = "../results/notebook/replies.sqlite"
sandbox_cache = "../results/notebook/sandbox.sqlite"
solver_model = solver("qwen2.5-coder:1.5b", "http://localhost:8000", cache)
mutator_model = mutator("qwen2.5-coder:7b")

# the study's search, 4 seeds then 8 generations of 4 mutated children
config = EvolutionConfig(
    mu=4, lambda_=4, generations=8, crossover_rate=0, seed=unit
)
log = RunLog(f"../results/notebook/unit{unit}.jsonl")

## Run

These are `aware-mixed`'s operators, every child comes from one of them at
random. Swap any piece, e.g. `Signal.BINARY`, another `Selector`, or other
names from `MUTATIONS`.

In [ ]:
train = HumanEval(sets["train"], sandbox_cache=sandbox_cache)
operators = MixedOperators(
    mutator_model,
    [MUTATIONS[name]() for name in
     ["rewrite", "first-order", "eda", "zero-order-hyper"]],
    train.description,
    seed=unit,
    log=log,
)
evolution = Evolution(
    config,
    Evaluator(train, solver_model, Signal.PARTIAL),
    operators,
    UniformSelector(),
    log,
)
population = evolution.run(SEED_PROMPTS)

best = population[0]
print(f"train score {best.evaluation.score:.3f}")
print(best.prompt)

## Validation pick and test set

Like the study, the pick is the best of the final population and the seeds
on the validation set. `val pick` minus `best seed` (also picked on
validation) is what evolution added, `no prompt` is the solver with an empty
system prompt.

In [ ]:
validator = Evaluator(
    HumanEval(sets["validation"], sandbox_cache=sandbox_cache),
    solver_model,
    Signal.PARTIAL,
)
candidates = [ind.prompt for ind in population + evolution.seeds]
picked, validation = pick_by_validation(validator, candidates)
log.validation(validation)

tester = Evaluator(
    HumanEval(sets["test"], sandbox_cache=sandbox_cache),
    solver_model,
    Signal.PARTIAL,
)
prompts = test_prompts(best, evolution.seeds, picked, validation)
tested = score_test_set(tester, prompts)
log.test(prompts, tested)
{name: evaluation.score for name, evaluation in tested.items()}